# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Evidence Notebook 08 - Core Error Analysis

Examiner-facing recovery notebook using only frozen Step-8 artifacts for the 44 answerable held-out test questions. It does not regenerate rankings or redesign the taxonomy.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display, Markdown

CC = Path(r"${THESIS_CONTROL_ROOT}")
WS = Path(r"${THESIS_WORKSPACE}")
SUMMARY_PATH = Path(r"${THESIS_CONTROL_ROOT}\Evidence_Recovery_and_Thesis_Assets") / "06_Validation" / "Notebook_Summaries" / "Evidence_Notebook_08_summary.json"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()

print("Evidence notebook mode: frozen-artifact verification only")
print("Correction Control:", CC)
print("Writing Workspace:", WS)


Evidence notebook mode: frozen-artifact verification only
Correction Control: ${THESIS_CONTROL_ROOT}
Writing Workspace: ${THESIS_WORKSPACE}


## Frozen Step-8 sources, population, and taxonomy

In [2]:
d = CC / "Core_Error_Analysis"
files = {
 "taxonomy": d / "RALSR_ERROR_TAXONOMY.csv",
 "recall": d / "RALSR_CANDIDATE_RECALL_SUMMARY.csv",
 "dense_fixed": d / "DENSE_VS_RALSR_OUTCOME_GROUPS.csv",
 "bm25_fixed": d / "BM25_VS_RALSR_OUTCOME_GROUPS.csv",
 "dense_bm25": d / "DENSE_VS_BM25_OUTCOME_GROUPS.csv",
 "coverage": d / "QUERY_SEMANTIC_COVERAGE_VS_OUTCOME.csv",
 "features": d / "RALSR_FEATURE_DIAGNOSTIC_SUMMARY.csv",
 "cases": d / "QUALITATIVE_CASES.csv",
}
display(pd.DataFrame([{"Source":k,"Path":str(v),"SHA256":sha256(v)} for k,v in files.items()]))
tax = pd.read_csv(files["taxonomy"])
assert int(tax.QID_Count.sum()) == 44
display(tax)

,Source,Path,SHA256
0,taxonomy,${THESIS_CODEX_ROOT}\Master Thesis - ...,01104F672DAAC7991341F53CCD5BEABA4A0C8575813674...
1,recall,${THESIS_CODEX_ROOT}\Master Thesis - ...,7B5F2800888095C6579DF9720783DEC7759DA0B53CB133...
2,dense_fixed,${THESIS_CODEX_ROOT}\Master Thesis - ...,5956499B754D505FA0B004064203815A792B1982EDAB4C...
3,bm25_fixed,${THESIS_CODEX_ROOT}\Master Thesis - ...,BBB3FE1FFCFDF6E652D25F3975D08B190B995CC6F7099B...
4,dense_bm25,${THESIS_CODEX_ROOT}\Master Thesis - ...,740D32FA1ABAC1157E7E2EFDCEF7A0165687C6989E1622...
5,coverage,${THESIS_CODEX_ROOT}\Master Thesis - ...,48F3902A595C02A1C5A643D377D0ED1664A99F473519EA...
6,features,${THESIS_CODEX_ROOT}\Master Thesis - ...,AF0A97AB338FF86D5D34B299367D59ED269E7A82578C86...
7,cases,${THESIS_CODEX_ROOT}\Master Thesis - ...,BE899A0558AEA99949641315AB27FAB24FC66876349A74...


,Primary_Category,Definition,QID_Count,Percentage_of_44,QIDs
0,Structural zero-candidate failure,No RALSR candidate passages exist.,2,0.045455,536;613
1,Candidate-generation failure,"Candidates exist, but no gold relevant passage...",6,0.136364,500;510;525;544;555;602
2,Candidate present but ranking failure,A gold relevant passage occurs in the candidat...,21,0.477273,505;508;513;521;527;529;539;540;541;542;545;55...
3,Partial retrieval success,At least one gold relevant passage occurs in t...,15,0.340909,502;518;519;528;531;533;534;550;552;557;559;57...
4,Strong retrieval success,Per-query AP@10 equals 1.0 exactly under the f...,0,0.000000,NaN


## Candidate recall, ranking failures, candidate-generation failures, structural failures, and successes

In [3]:
recall = pd.read_csv(files["recall"])
display(recall)
counts = dict(zip(tax.Primary_Category, tax.QID_Count))
assert counts["Structural zero-candidate failure"] == 2
assert counts["Candidate-generation failure"] == 6
assert counts["Candidate present but ranking failure"] == 21
assert counts["Partial retrieval success"] == 15
expected_recall = {
 "At least one gold passage in full candidate set":36,
 "At least one gold passage in fixed Top 100":30,
 "At least one gold passage in fixed Top 10":15,
 "At least one gold passage at fixed rank 1":4,
}
for k,v in expected_recall.items():
    got = int(recall.loc[recall.Measure.eq(k),"Count"].iloc[0]); assert got == v
print("Frozen candidate-recall decomposition: PASS")

,Measure,Count,Denominator,Percentage
0,Structural zero-candidate failures,2.000000,44.0,0.045455
1,At least one gold passage in full candidate set,36.000000,44.0,0.818182
2,No gold passage in full candidate set,8.000000,44.0,0.181818
3,At least one gold passage in fixed Top 100,30.000000,44.0,0.681818
4,At least one gold passage in fixed Top 10,15.000000,44.0,0.340909
5,At least one gold passage at fixed rank 1,4.000000,44.0,0.090909
6,Candidate-set size mean,296.113636,NaN,NaN
7,Candidate-set size median,231.000000,NaN,NaN
8,Candidate-set size minimum,0.000000,NaN,NaN
9,Candidate-set size maximum,1102.000000,NaN,NaN


Frozen candidate-recall decomposition: PASS


## Pairwise system outcomes

These groups are loaded directly from Step 8 and are not recategorized.

In [4]:
pairwise = {}
for key in ["dense_fixed","bm25_fixed","dense_bm25"]:
    pairwise[key] = pd.read_csv(files[key])
    display(pairwise[key])
assert pairwise["dense_fixed"].QID_Count.tolist() == [12,6,9,17]
assert pairwise["bm25_fixed"].QID_Count.tolist() == [10,9,6,19]
assert pairwise["dense_bm25"].QID_Count.tolist() == [12,7,9,16]

,Comparison,Outcome_Group,QID_Count,Percentage_of_44,QIDs
0,Dense E5 vs Fixed RALSR,Dense E5-only success,12,0.272727,505;513;525;527;529;545;563;568;571;576;577;611
1,Dense E5 vs Fixed RALSR,Fixed RALSR-only success,6,0.136364,518;528;531;534;550;559
2,Dense E5 vs Fixed RALSR,Both succeed,9,0.204545,502;519;533;552;557;570;575;580;610
3,Dense E5 vs Fixed RALSR,Both fail,17,0.386364,500;508;510;521;536;539;540;541;542;544;555;55...


,Comparison,Outcome_Group,QID_Count,Percentage_of_44,QIDs
0,BM25 vs Fixed RALSR,BM25-only success,10,0.227273,505;529;539;540;541;545;560;562;568;571
1,BM25 vs Fixed RALSR,Fixed RALSR-only success,9,0.204545,519;528;533;534;550;552;559;570;610
2,BM25 vs Fixed RALSR,Both succeed,6,0.136364,502;518;531;557;575;580
3,BM25 vs Fixed RALSR,Both fail,19,0.431818,500;508;510;513;521;525;527;536;542;544;555;55...


,Comparison,Outcome_Group,QID_Count,Percentage_of_44,QIDs
0,Dense E5 vs BM25,Dense E5-only success,12,0.272727,513;519;525;527;533;552;563;570;576;577;610;611
1,Dense E5 vs BM25,BM25-only success,7,0.159091,518;531;539;540;541;560;562
2,Dense E5 vs BM25,Both succeed,9,0.204545,502;505;529;545;557;568;571;575;580
3,Dense E5 vs BM25,Both fail,16,0.363636,500;508;510;521;528;534;536;542;544;550;555;55...


## Coverage associations, root-source evidence, and feature discrimination

Associations are descriptive; they are not treated as causal. Frozen feature diagnostics compare the highest-ranked gold candidate with the top nonrelevant candidate where ranking analysis is possible.

In [5]:
coverage = pd.read_csv(files["coverage"])
features = pd.read_csv(files["features"])
display(coverage)
display(features)
cases = pd.read_csv(files["cases"])
display(cases.head(12))

,Semantic_Coverage_Bin,QID_Count,Mean_Fixed_RALSR_AP_at_10,Mean_Fixed_RALSR_RR,Structural_Zero_Candidate_Count,Candidate_Generation_Failure_Count,Candidate_Generation_Failure_Rate,Ranking_Failure_Count,Ranking_Failure_Rate,Small_Sample_Flag
0,100%,6,0.177801,0.430556,0,0,0.000000,2,0.333333,False
1,50–<100%,17,0.017400,0.135854,0,2,0.117647,9,0.529412,False
2,>0–<50%,21,0.039292,0.095238,2,4,0.190476,10,0.476190,False
3,0%,0,NaN,NaN,0,0,NaN,0,NaN,True


,Feature,Ranking_Failure_QID_Count,Mean_Gold_Minus_Top_Nonrelevant,Median_Gold_Minus_Top_Nonrelevant,Gold_Higher_Count,Equal_Count,Gold_Lower_Count
0,Root_Coverage,21,-0.210150,-0.233895,0,4,17
1,Root_Overlap,21,-0.132275,-0.111111,0,4,17
2,Semantic_Overlap,21,-0.102041,-0.142857,0,7,14
3,Jaccard_Root_Similarity,21,-0.092626,-0.074190,5,0,16
4,Passage_Root_Count,21,-0.135854,-0.176471,6,1,14
5,Semantic_Count,21,-0.134199,-0.204545,7,0,14


,QID,Case_Groups,Question,Selected_Terms,Accepted_Roots,Unresolved_Terms,Semantic_Coverage_Proportion,Gold_Relevant_Passage_IDs,Gold_Passage_Excerpts,BM25_Top_Relevant_Rank,BM25_Top_Relevant_Rank_Top100,Dense_Top_Relevant_Rank,Dense_Top_Relevant_Rank_Top100,Fixed_RALSR_Top_Relevant_Rank,Tuned_RALSR_Top_Relevant_Full_Candidate_Rank,RALSR_Candidate_Set_Inclusion,Highest_Ranked_Gold_Six_Normalized_Features,Query_Root_Corpus_Passage_Hit_Counts,Evidence_Grounded_Interpretation
0,510,"C — Both fail, most uncovered terms; F — Candi...","اتفاقية """"سيداو"""" تقول لا يُسمح للرجل بتعدد ال...","[""اتفاقية"", ""سيداو"", ""تقول"", ""يسمح"", ""للرجل"", ...","[""قول"", ""سمح"", ""رجل"", ""عدد"", ""ردد"", ""نصص"", ""قرن""]","[""اتفاقية"", ""سيداو"", ""الزوجات""]",0.400000,"[""4:2-6""]",4:2-6: وآتوا اليتامى أموالهم ولا تتبدلوا الخبي...,NaN,NaN,NaN,97.0,NaN,NaN,False,NaN,"{""قول"": 0, ""سمح"": 0, ""رجل"": 64, ""عدد"": 149, ""ر...","RALSR generated candidates, but no judged rele..."
1,518,B — RALSR-only success,لكون البخل لا يقتصر على المال فقط، ماذا يشمل ا...,"[""لكون"", ""البخل"", ""يقتصر"", ""المال"", ""فقط"", ""يش...","[""بخل"", ""قصر"", ""مول"", ""فقط"", ""شمل"", ""شحح"", ""بم""]","[""لكون"", ""فيه""]",0.444444,"[""107:1-3"", ""107:4-7"", ""17:97-100"", ""33:18-20""...",107:1-3: أرأيت الذي يكذب بالدين. فذلك الذي يدع...,4.0,4.0,NaN,15.0,4.0,4.0,True,"{""Root_Coverage"": 0.24810526315789477, ""Root_O...","{""بخل"": 7, ""قصر"": 10, ""مول"": 0, ""فقط"": 0, ""شمل...",The root-aware system retrieved judged evidenc...
2,525,F — Candidate-generation failure,ماذا تشمل الأمانة في القرآن؟,"[""تشمل"", ""الامانة"", ""القران""]","[""شمل"", ""امن"", ""قرن""]",[],0.666667,"[""11:84-88"", ""12:11-15"", ""12:50-57"", ""23:1-11""...",11:84-88: وإلى مدين أخاهم شعيبا قال يا قوم اعب...,NaN,73.0,1.0,1.0,NaN,NaN,False,NaN,"{""شمل"": 9, ""امن"": 0, ""قرن"": 91}","RALSR generated candidates, but no judged rele..."
3,534,B — RALSR-only success,الحلي من الذهب لا يجوز أن يلبسها الرجل في الدن...,"[""الحلي"", ""الذهب"", ""يجوز"", ""يلبسها"", ""الرجل"", ...","[""حلي"", ""ذهب"", ""لبس"", ""رجل"", ""دنو"", ""لبس""]","[""يجوز"", ""سيكون"", ""مباحا"", ""الآخرة""]",0.400000,"[""18:29-31"", ""22:23-24"", ""35:29-35""]",18:29-31: وقل الحق من ربكم فمن شاء فليؤمن ومن ...,NaN,NaN,NaN,NaN,1.0,1.0,True,"{""Root_Coverage"": 0.368421052631579, ""Root_Ove...","{""حلي"": 0, ""ذهب"": 49, ""لبس"": 17, ""رجل"": 64, ""د...",The root-aware system retrieved judged evidenc...
4,536,E — Structural false abstention,هل يريد الله منا أن نكون قانتين؟,"[""يريد"", ""الله"", ""منا"", ""نكون"", ""قانتين""]","[""رود"", ""اله"", ""منن"", ""كون""]","[""قانتين""]",0.400000,"[""2:238-239"", ""33:28-31"", ""3:42-44"", ""4:34-35""]",2:238-239: حافظوا على الصلوات والصلاة الوسطى و...,NaN,NaN,NaN,30.0,NaN,NaN,False,NaN,"{""رود"": 0, ""اله"": 0, ""منن"": 0, ""كون"": 0}",All selected linguistic evidence failed to pro...
5,545,A — Dense-only success; D — Candidate present ...,هل هناك فرق بين علم اليقين وعين اليقين؟,"[""هناك"", ""فرق"", ""علم"", ""اليقين"", ""وعين""]","[""فرق"", ""علم"", ""يقن""]","[""هناك"", ""وعين""]",0.600000,"[""102:5-8""]",102:5-8: كلا لو تعلمون علم اليقين. لترون الجحي...,4.0,4.0,1.0,1.0,131.0,24.0,True,"{""Root_Coverage"": 0.2982105263157895, ""Root_Ov...","{""فرق"": 54, ""علم"": 543, ""يقن"": 0}",A judged relevant passage was available to RAL...
6,559,B — RALSR-only success,من هم الذين حاولوا التحايل على حكم الله المتع...,"[""حاولوا"", ""التحايل"", ""حكم"", ""الله"", ""المتعلق""...","[""حكم"", ""اله"", ""علق""]","[""حاولوا"", ""التحايل"", ""بالأوقات"", ""المباحة"", ""...",0.250000,"[""16:120-124"", ""2:63-66"", ""4:153-161"", ""4:47-4...",16:120-124: إن إبراهيم كان أمة قانتا لله حنيفا...,NaN,NaN,NaN,62.0,2.0,3.0,True,"{""Root_Coverage"": 0.2982105263157895, ""Root_Ov...","{""حكم"": 171, ""اله"": 0, ""علق"": 6}",The root-aware system retrieved judged evidenc...
7,560,"C — Both fail, most uncovered terms",المصلح هو الصالح في ذاته والمصلح لغيره. ما الد...,"[""المصلح"", ""الصالح"", ""ذاته"", ""وا

## Representative Dense-only and RALSR-complementary cases

The qualitative cases were selected by the predeclared Step-8 procedure. The structural false abstentions are qids 536 and 613.

In [6]:
summary = {
 "answerable_test_qids":44,"structural_zero":2,"candidate_generation_failure":6,
 "ranking_failure":21,"top10_success":15,"candidate_recall_full":36,
 "candidate_recall_top100":30,"candidate_recall_top10":15,"rank1_success":4,
 "dense_only":12,"ralsr_only":6,"both_success":9,"both_fail":17,
 "substantive_mismatches":0,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Ranking discrimination is the largest observed fixed-RALSR bottleneck, followed by candidate exclusion.")
print("Substantive mismatch count: 0")

Ranking discrimination is the largest observed fixed-RALSR bottleneck, followed by candidate exclusion.
Substantive mismatch count: 0


## Final interpretation and thesis references

Use the Step-12 RALSR error-decomposition figure and error-taxonomy/pairwise table. The evidence supports a descriptive bottleneck claim only; coverage and representation gaps are not proven causes.